E-13: compatibility and tangent-direction balance (H13, `prop:arbitrary_pair_foc`)

Registration: `doc/2026-10-07_experiment_registration.md` (parent repository), E-13, §1.3 A-4 and §1.4–§1.9, including the rule (i')–(iv') for incompatible pairs; Amendment 4 for the Newton acceptance. One notebook for every stage; the runner passes the stage:

- `python3 notebooks/experiments/run_registered.py E-13 stage0 11_E13_compatibility_tangent_balance.ipynb`
- `... E-13 stage0.5 ...`, then `... E-13 stage1 ...` (each after the previous stage is committed).

DGP, pairs, the replication function and the aggregation are in `grrexp/e13.py`; the pilot and the confirmatory stage run the same aggregation. A solver that raises stops the notebook and the run is recorded as failed (§1.1).

In [1]:
import io
import json
import os
import time

import pandas as pd

from grrexp import env, outputs

gr = env.use_submodule_genriesz()
from grrexp import baselines, e12, e13, parallel  # noqa: E402
from grrexp.seeds import CONFIRMATORY_ENTROPY, PILOT_ENTROPY, Seeds  # noqa: E402

STAGE = json.loads(os.environ[outputs.RUN_ENV])["stage"]
rec = outputs.RunRecorder(13, STAGE, cells=None if STAGE == "stage0" else e13.CELLS,
                          arms=None if STAGE == "stage0" else [a.replace("|", "-") for a in e13.ARM_LABELS])
STAGE

'stage0.5'

# Stage 0: population quantities and certificates (registration E-13 Stage 0, §1.9)

In [2]:
if STAGE == "stage0":
    dgp = e13.dgp_check()
    print(json.dumps(dgp, indent=1))
    if not dgp["ok"]:
        raise RuntimeError("DGP13 does not reproduce the registered range of e and E alpha0^2")
    rows, stage0_warnings, _ = baselines.run_recording_warnings(e13.stage0)
    rec.write_stage0("e13_population", {"pairs": rows, "dgp_check": dgp})
    for r in rows:
        print(r["pair"], r["certified"], r["checks"])
    rec.finalize(R=None, n=None, warnings=int(sum(stage0_warnings.values())), failures=0)

# The registered workflow after the replications (shared by Stage 0.5 and Stage 1)

H13 (a)(b) (stops on a violation), the §1.5 metrics, the family-1 MCSE intervals, and the family H13 (c)–(e) (Holm 0.01).

Bootstrap call order (stream 4, one generator per family). Family 1 (descriptive MCSE, `e12.family1_mcse`): the rows of the summary in order (cells in registered order, then `e13.ARM_LABELS`), within a row CI length (mean), max weight (median), SE ratio; rows with fewer than two successes consume no draws. Family 2 (the se of H13 (c)): the pairs I-SQexp, I-BKLexp, I-UKLlin (certified ones only), then $j = 0, \dots, 7$, one call each on the successful replications of $n = 4000$; a comparison that is unavailable (fewer than two successes) consumes no draws.

In [3]:
def load_population():
    return json.loads((outputs.stage_dir(13, "stage0") / "stage0_e13_population.json").read_text())["pairs"]


def aggregate(raw, pop, seeds):
    bal = e13.balance_checks(raw)
    summ = e13.summarise(raw, pop)
    summ = summ.assign(**e12.family1_mcse(raw, summ, seeds))
    verdict, tests, unavailable = e13.family_h13(summ, raw, seeds)
    summ["family_verdict"] = json.dumps(verdict.sentence() if verdict else None)
    summ["family_tests"] = json.dumps(tests)
    summ["family_rejected"] = json.dumps(list(verdict.rejected) if verdict else [])
    summ["unavailable_tests"] = json.dumps(unavailable)
    summ["h13_balance"] = json.dumps(bal)
    return summ

# Stage 0.5: timing of the complete workflow, status counts, 1-versus-12-worker identity (no estimates kept)

Per cell and number of workers: the 10 replications and the aggregation of the cell. The aggregation of the whole pilot sample and the table and figure generation are timed once and recorded in the manifest. Pilot estimates, tables and figures are discarded.

In [4]:
if STAGE == "stage0.5":
    import matplotlib.pyplot as plt

    parallel.configure_worker()
    pop = load_population()
    timing, serial, para, all_tasks = [], [], [], []
    for workers in (1, outputs.PILOT_WORKERS):
        for ci, cell in enumerate(e13.CELLS):
            t0 = time.perf_counter()
            tasks = e13.tasks_for(PILOT_ENTROPY, outputs.PILOT_REPS, cells=[ci])
            res = parallel.run_tasks(e13.replicate, tasks, workers)
            aggregate(e13.raw_frame(tasks, res), pop, Seeds(13, entropy=PILOT_ENTROPY))
            timing.append({"cell": cell, "workers": workers, "seconds": time.perf_counter() - t0})
            if workers == 1:
                serial += res
                all_tasks += tasks
            else:
                para += res
    digest = parallel.assert_bit_identical(serial, para, expected=len(all_tasks))
    raw = e13.raw_frame(all_tasks, serial)
    t0 = time.perf_counter()
    buf = io.StringIO()
    aggregate(raw, pop, Seeds(13, entropy=PILOT_ENTROPY)).to_csv(buf, index=False)
    buf.seek(0)
    S = pd.read_csv(buf)
    e13.tables(S)
    plt.close(e13.figure(S))
    whole_seconds = time.perf_counter() - t0
    counts = raw.groupby(["cell", "arm", "status"]).size().reset_index(name="count")
    status_df = pd.DataFrame({"cell": [e13.CELLS[c] for c in counts["cell"]],
                              "arm": counts["arm"].str.replace("|", "-", regex=False),
                              "status": counts["status"], "count": counts["count"].astype(int)})
    rec.write_pilot("timing.csv", pd.DataFrame(timing))
    rec.write_pilot("status_counts.csv", status_df)
    rec.record_parallel_identity(cells=e13.CELLS, reps=outputs.PILOT_REPS, n_jobs=outputs.PILOT_WORKERS,
                                 digest=digest)
    print(pd.DataFrame(timing).groupby("workers")["seconds"].sum(), whole_seconds)
    print(status_df.groupby(["arm", "status"])["count"].sum().to_string())
    rec.finalize(R={"main": outputs.PILOT_REPS}, n={"main": list(e13.N_VALUES)},
                 warnings=e13.total_warnings(raw), failures=int((raw["status"] != "ok").sum()),
                 extra={"whole_pilot_aggregation_table_and_figure_seconds": whole_seconds})
    del raw

workers
1     10.581695
12     5.662982
Name: seconds, dtype: float64 0.16228891699574888
arm               status                        
C-BKL1-ARW_cf     domain_prediction                 12
                  ok                                 8
C-BKL1-RW_full    domain_prediction                  5
                  ok                                15
C-SQ-ARW_cf       ok                                20
C-SQ-RW_full      ok                                20
C-UKL1-ARW_cf     ok                                20
C-UKL1-RW_full    ok                                20
I-BKLexp-ARW_cf   ok                                20
I-BKLexp-RW_full  ok                                20
I-SQexp-ARW_cf    linesearch                        13
                  ok                                 7
I-SQexp-RW_full   linesearch                         5
                  nonfinite                          1
                  ok                                14
I-UKLlin-ARW_cf   uncertified_numeri

# Stage 1: confirmatory Monte Carlo

In [5]:
if STAGE == "stage1":
    parallel.configure_worker()
    tasks = e13.tasks_for(CONFIRMATORY_ENTROPY, e13.R)
    t0 = time.perf_counter()
    results = parallel.run_tasks(e13.replicate, tasks, outputs.PILOT_WORKERS)
    print("seconds", time.perf_counter() - t0)
    raw = e13.raw_frame(tasks, results)
    rec.write_raw(raw)
    n_warn = e13.total_warnings(raw)
    raw.shape, n_warn

In [6]:
if STAGE == "stage1":
    summ = aggregate(raw, load_population(), Seeds(13))  # H13 (a)(b) stop the notebook on a violation
    rec.write_summary(summ)
    {k: json.loads(summ[k].iloc[0]) for k in ("h13_balance", "family_verdict", "family_rejected", "unavailable_tests")}

# Table and figure (from summary.csv only, §4)

In [7]:
if STAGE == "stage1":
    S = outputs.read_summary(rec)
    tabs, fam = e13.tables(S)
    for name, tex in tabs.items():
        rec.write_table(name, tex)
    rec.save_figure(e13.figure(S), "fig_E13_tangent")
    print(fam)
    rec.finalize(R={"main": e13.R}, n={"main": list(e13.N_VALUES)}, warnings=int(n_warn),
                 failures=int((S["R"] - S["R_s"]).sum()))